In [1]:
from google.colab import drive
drive.mount('/content/drive')

# cd to the *root* of career-ml, NOT to scripts
%cd "/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml"

Mounted at /content/drive
/content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml


In [2]:
from pathlib import Path
import pandas as pd
import re


# ---------- paths ----------
try:
    BASE_DIR = Path(__file__).resolve().parents[1]
except NameError:
    BASE_DIR = Path.cwd()

PROCESSED_DIR = BASE_DIR / "data" / "processed"
ALL_JOBS_PATH = PROCESSED_DIR / "all_jobs_tagged.csv"
OUT_PATH = PROCESSED_DIR / "jobs_unified.csv"


# ---------- helpers ----------

def choose_first_nonempty(row, cols):
    for c in cols:
        if c in row.index:
            val = row[c]
            if isinstance(val, str) and val.strip():
                return val
    return ""


def normalize_spaces(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def main():
    if not ALL_JOBS_PATH.exists():
        raise FileNotFoundError(f"{ALL_JOBS_PATH} not found")

    df = pd.read_csv(ALL_JOBS_PATH, low_memory=False)

    # --- pick unified title ---
    title_candidates = ["job_name", "Job Title", "title", "name"]
    df["job_title"] = df.apply(lambda r: choose_first_nonempty(r, title_candidates), axis=1)
    df["job_title"] = df["job_title"].astype(str).apply(normalize_spaces)

    # --- main description text ---
    desc_candidates = ["description", "Job Description", "three_reasons"]
    df["description"] = df.apply(lambda r: choose_first_nonempty(r, desc_candidates), axis=1)
    df["description"] = df["description"].astype(str).apply(normalize_spaces)

    # --- requirements / skills text ---
    req_candidates = ["requirment", "r", "Required Skills", "taglist"]
    df["requirements_text"] = df.apply(lambda r: choose_first_nonempty(r, req_candidates), axis=1)
    df["requirements_text"] = df["requirements_text"].astype(str).apply(normalize_spaces)

    # --- experience / level ---
    exp_candidates = ["experience", "Experience Level"]
    df["experience_raw"] = df.apply(lambda r: choose_first_nonempty(r, exp_candidates), axis=1)
    df["experience_raw"] = df["experience_raw"].astype(str).apply(normalize_spaces)

    # --- salary ---
    sal_candidates = ["salary_range", "Salary Range"]
    df["salary_raw"] = df.apply(lambda r: choose_first_nonempty(r, sal_candidates), axis=1)
    df["salary_raw"] = df["salary_raw"].astype(str).apply(normalize_spaces)

    # --- job type ---
    jobtype_candidates = ["Job Type", "job_type"]
    df["job_type"] = df.apply(lambda r: choose_first_nonempty(r, jobtype_candidates), axis=1)
    df["job_type"] = df["job_type"].astype(str).apply(normalize_spaces)

    # --- industry (only present in UK dataset, optional but useful) ---
    industry_candidates = ["Industry", "industry"]
    df["industry"] = df.apply(lambda r: choose_first_nonempty(r, industry_candidates), axis=1)
    df["industry"] = df["industry"].astype(str).apply(normalize_spaces)

    # --- build a unified text field for modelling (description + requirements) ---
    df["job_text"] = (
        df["description"].fillna("") + " " +
        df["requirements_text"].fillna("")
    ).apply(normalize_spaces)

    # --- columns we keep ---
    keep_cols = [
        # core identifiers
        "job_uid",
        "source",

        # unified semantic columns
        "job_title",
        "description",
        "requirements_text",
        "job_text",

        # skill tagging
        "matched_skill_ids",
        "matched_skills",
        "matched_skill_count",

        # extra context (optional for modelling, good for analysis)
        "experience_raw",
        "salary_raw",
        "job_type",
        "industry",
    ]

    # Some may not exist (e.g. job_uid if earlier scripts changed), so filter safely
    keep_cols = [c for c in keep_cols if c in df.columns]

    unified = df[keep_cols].copy()

    # drop rows without a title (they are not useful for role prediction)
    unified = unified[unified["job_title"].astype(str).str.strip() != ""].reset_index(drop=True)

    unified.to_csv(OUT_PATH, index=False)
    print(f"Saved unified jobs dataset with {len(unified)} rows to {OUT_PATH}")
    print("Columns:", list(unified.columns))


if __name__ == "__main__":
    main()


Saved unified jobs dataset with 15230 rows to /content/drive/MyDrive/SLIIT/Research/Uni-Finder/career-ml/data/processed/jobs_unified.csv
Columns: ['job_uid', 'source', 'job_title', 'description', 'requirements_text', 'job_text', 'matched_skill_ids', 'matched_skills', 'matched_skill_count', 'experience_raw', 'salary_raw', 'job_type', 'industry']
